ass

crear adm_obj_tg.RhfEfectivaN RHF VARCHAR(3),[SOLO FH1]INT,[SOLO FH2]INT,[SOLO FH3]INT,[DOBLE FH]INT,[TRIPLE FH] INT
crear adm_obj_tg.NumxDnixEfectivaN |NUMDOCUMENTO as DNI,count(*) as numxdni

crear adm_obj_tg.FunnelEfectivaN -->
    tabla
        [ADM_OBJ_TG].[synBaseMaestraEfectivaNegocios]
        [ADM_OBJ_TG].[synTmpLLamadasEfectivaN]
        adm_obj_tg.NumxDnixEfectivaN
        [ADM_OBJ_TG].[synUsuariosTg]
        adm_obj_tg.RhfEfectivaN
    where a.NUMDOCUMENTO IS NOT NULL

actualizo adm_obj_tg.RhfEfectivaN 
    columnas 
        TIPO_GESTION  ='CONTACTO EFECTIVO', 
        GESTION ='EFECTIVO', 
        SUBGESTION ='VOLVER A LLAMAR'
    tabla 
        [ADM_OBJ_TG].[synVentasEfectivaN] left
    where b.dni is null and a.SUBGESTION='SI QUIERE'

actualizo adm_obj_tg.RhfEfectivaN 
    columnas 
        TIPO_GESTION  ='CONTACTO EFECTIVO', 
        GESTION ='EFECTIVO', 
        SUBGESTION ='VOLVER A LLAMAR'
    tabla [ADM_OBJ_TG].[synVentasEfectivaN] inner

actualizo adm_obj_tg.RhfEfectivaN 
    columnas Ejecutivo tNombreCompleto
    tabla [ADM_OBJ_TG].[synVentasEfectivaN] inner
          [ADM_OBJ_TG].[synUsuariosTg] inner

actualizo adm_obj_tg.RhfEfectivaN 
    columnas MntOferta = monto
    tabla [ADM_OBJ_TG].[synVentasEfectivaN] inner

actualizo adm_obj_tg.RhfEfectivaN 
    columnas Estado = isnull(replace(b.ESTADO,'-','EN PROCESO'),'EN PROCESO')
    tabla [ADM_OBJ_TG].[synVentasEfectivaN] inner

actualizo adm_obj_tg.RhfEfectivaN 
    columnas CntEstado=1
    tabla [ADM_OBJ_TG].[synVentasEfectivaN] inner
    where ESTADO='1.-VALIDADA'

actualizo adm_obj_tg.RhfEfectivaN 
    columnas FECHA_LLAMADA =convert(varchar,CONVERT(date,b.FECHA))
    , a.DIA=substring(convert(varchar,b.FECHA),9,2)
    tabla [ADM_OBJ_TG].[synVentasEfectivaN] inner

actualizo adm_obj_tg.RhfEfectivaN 
    columnas a.[NUM_DIA_HABIL]=b.[NUM_DIA_HABIL]
    a.Semana_Mes=b.Semana_Mes 
    tabla [ADM_OBJ_TG].[synDiaHabil] inner

crear adm_obj_tg.LlamadasEfectivaN1
    group by LlamadasEfectivaN1
        columnas DNI AS DNI_LLAMADAS
        , COUNT(*) AS CNT_LLAMADAS1 
    tabla [ADM_OBJ_TG].[synTmpLlamadasGeneralEfectivaN] 


crear adm_obj_tg.tDesembolso_efeneg 
    tabla
        [ADM_OBJ_TG].[DesembolsoNegociosEfe] 
    where DATENAME(MONTH,FECHA_DESEMBOLSO)=DATENAME(MONTH,GETDATE()) and YEAR(FECHA_DESEMBOLSO)=YEAR(getdate());

actualizar adm_obj_tg.FunnelEfectivaN
    columna        
        a.tMontoDesem =b.MONTO_NETO
    tabla 
        tDesembolso_efeneg

eliminar info del mes actual [ADM_OBJ_TG].[tGestionMesEfectivaN]
    tabla
        adm_obj_tg.FunnelEfectivaN
        adm_obj_tg.LlamadasEfectivaN1


-----------------------------------------------
tambien hay q preparar el resumen


In [ ]:
select a.*  from (
    select 
        tMesGestion,NUMERO_DOCUMENTO+' '+convert(varchar,fecha_envio) llave,'Entregados' MedicionDatos,sum(NRG) Q,AÑO_DURACION_BASE 
    from [ADM_OBJ_TG].[tGestionMesEfectivaN] 
    where NRG=1 
    GROUP BY NUMERO_DOCUMENTO,FECHA_ENVIO,tMesGestion,AÑO_DURACION_BASE
    UNION ALL
    select 
        tMesGestion,NUMERO_DOCUMENTO+' '+convert(varchar,fecha_envio) llave,'Recorridos' MedicionDatos,sum(RECORRIDO) Q,AÑO_DURACION_BASE 
    from [ADM_OBJ_TG].[tGestionMesEfectivaN] 
    where RECORRIDO=1 
    GROUP BY NUMERO_DOCUMENTO,FECHA_ENVIO,tMesGestion,AÑO_DURACION_BASE
    UNION ALL
    select 
        tMesGestion,NUMERO_DOCUMENTO+' '+convert(varchar,fecha_envio) llave,'Llamados' MedicionDatos,sum(CNT_LLAMADAS) Q,AÑO_DURACION_BASE 
    from [ADM_OBJ_TG].[tGestionMesEfectivaN] 
    where RECORRIDO=1 
    GROUP BY NUMERO_DOCUMENTO,FECHA_ENVIO,tMesGestion,AÑO_DURACION_BASE
    UNION ALL
    select 
        tMesGestion,NUMERO_DOCUMENTO+' '+convert(varchar,fecha_envio) llave,'Titular' MedicionDatos,sum(CET) Q,AÑO_DURACION_BASE 
    from [ADM_OBJ_TG].[tGestionMesEfectivaN] 
    where CET=1 
    GROUP BY NUMERO_DOCUMENTO,FECHA_ENVIO,tMesGestion,AÑO_DURACION_BASE
    UNION ALL
    select 
        tMesGestion,NUMERO_DOCUMENTO+' '+convert(varchar,fecha_envio) llave,'Ventas' MedicionDatos,sum(CNTVTAS) Q,AÑO_DURACION_BASE 
    from [ADM_OBJ_TG].[tGestionMesEfectivaN] 
    where CNTVTAS=1 
    GROUP BY NUMERO_DOCUMENTO,FECHA_ENVIO,tMesGestion,AÑO_DURACION_BASE 
) a where tMesGestion=DATENAME(MONTH,GETDATE()) and AÑO_DURACION_BASE='2025';


select vnt.*,
cast(dt.NUM_DIA_HABIL as int) dia_,dt.[Semana_Mes],tp.[tSupervisor]
into [ADM_OBJ_TG].VentasEfectivaNFunnel
from [ADM_OBJ_TG].[synVentasEfectivaNFunnel] VNT 
INNER JOIN [ADM_OBJ_TG].[synDiaHabil] DT ON VNT.FECHA=DT.Fecha
inner join [ADM_OBJ_TG].[synUsuariosTg] tp on vnt.DNIEjecutivo=tp.tDocumentoVici
WHERE DATENAME(MONTH,VNT.FECHA) in (@MES1,@MES2,@MES3)
and CAMPANA='Negocios'
order by FECHA


In [ ]:
ssss

In [2]:

import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *
from unidecode import unidecode
from sqlalchemy import create_engine
from sqlalchemy import text

fecha_mes_base='2026-06-01'
tipi_cond1='RECLUTAMIENTO'
servidor_01=64

campana='negocios'

server_sql = server_kishin
db_sql = "DANTALION"
user_sql = user_kishin
pwd_sql = pwd_kishin

engine_kishin = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)
server_sql = server_zeus
db_sql = "ODIN"
user_sql = user_zeus
pwd_sql = pwd_zeus
engine_zeus = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)
server_sql = server_sa
db_sql = "ODIN"
user_sql = user_sa
pwd_sql = pwd_sa
engine_sa = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)
server_sql = server_zeus
db_sql = "SAMANTHA"
user_sql = user_zeus
pwd_sql = pwd_zeus
engine_samantha = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)

In [ ]:
from sqlalchemy import text

with engine_samantha.begin() as conn:
    conn.execute(text("delete from SAMANTHA.dbo.efectiva_negocios_ventas where tipo_venta_ref is not null"))

query = """
  SELECT 
      A.FECHA,
      A.dni,
      A.CLIENTE,
      A.PROMOTOR,
      A.ESTADO,
      A.AUDITOR,
      A.MES_VENTA,
      A.CAMPANA,
      A.DIA,
      A.TRAMA_HORA,
      A.CANDADO,
      A.MONTO,
      A.EJECUTIVO,
      A.TRAMA_HORA_Val,
      A.ANIO,
      A.DNIEjecutivo,
      A.celular,
      'target' as tipo_venta_ref,
      R.fecha_ref
  FROM SAMANTHA.dbo.Ventas_Target A
  CROSS JOIN (
      SELECT MAX(FECHA) AS fecha_ref
      FROM SAMANTHA.dbo.efectiva_negocios_ventas
      WHERE CAMPANA = 'negocios'
        AND FECHA >= '2026-06-01'
  ) R
  WHERE A.CAMPANA = 'negocios'
    AND A.FECHA >= '2026-06-01'
    AND NOT EXISTS (
          SELECT 1
          FROM SAMANTHA.dbo.efectiva_negocios_ventas B
          WHERE B.DNI = A.DNI COLLATE Modern_Spanish_CI_AS
            AND B.FECHA >= '2026-06-01'
    )
  """
df_venta_target= pd.read_sql(query,engine_zeus)

df_venta_target["FECHA"] = pd.to_datetime(df_venta_target["FECHA"])
df_venta_target = (
    df_venta_target
    .sort_values("FECHA", ascending=False)
    .drop_duplicates(subset="dni", keep="first")
    .reset_index(drop=True)
)

df_venta_target = df_venta_target[
    (df_venta_target["fecha_ref"].isna()) |
    (df_venta_target["FECHA"] > df_venta_target["fecha_ref"])
].reset_index(drop=True)
df_venta_target=df_venta_target.drop(columns='fecha_ref')

if not df_venta_target.empty:
    df_venta_target.to_sql(
        name="efectiva_negocios_ventas",
        con=engine_samantha,
        if_exists="append",
        index=False,
        chunksize=1000
    )
    print('ok')



In [38]:
from sqlalchemy import text

with engine_zeus.begin() as conn:
    conn.execute(text("truncate table ODIN.dbo.tNumeroEfectivaNegocios"))

query = f"""
    select NUMDOCUMENTO as DNI,count(*) as NUMXDNI 
    from ODIN.dbo.tNumeroEfectivaNegocios 
    group by NUMDOCUMENTO
"""
df_num_dni= pd.read_sql(query, engine_zeus)

if not df_num_dni.empty:
    df_num_dni.to_sql(
        name="NumxDnixEfectivaN",
        con=engine_zeus,
        if_exists="replace",
        index=False,
        chunksize=1000
    )
    print('ok')



In [19]:
query = """
SELECT 
	A.NUMDOCUMENTO as NUMERO_DOCUMENTO,
	CASE WHEN A.NUMDOCUMENTO IS NULL THEN B.Dni ELSE NULL END AS DNI_VICI, 
	CASE WHEN A.NUMDOCUMENTO IS NULL THEN ven.Dni ELSE NULL END AS DNI_VEN, 
	CASE WHEN A.NUMDOCUMENTO IS NULL THEN desem.Dni ELSE NULL END AS DNI_DES, 
	CASE
		WHEN A.NUMDOCUMENTO IS NULL THEN 'FUERA DE BASE'
		ELSE 'EN BASE'
	END as TIPO_BASE,
	a.SEGMICROPEQUENACOMERCIAL as Prioridad,
	a.TIPOBASEMICROPEQUENA,
    a.DEPARTAMENTO,
    a.PROVINCIA,
    a.DISTRITO,
    a.AREAEFECTINEGOCIOS,
    a.num_cuotas_pagadas,
    a.prioridad,
    a.PERFILINICIAL,
    a.ULTIMOMONTODESEMBOLSADO,
    a.ULTIMOMONTODESEMBOLSADOPLUS20,
    a.flag_recurrencia_efectinegocio,
    a.RETIRO,
    a.DEVUELTO,
    a.recompra,
    a.Proceso, --MARCA
    a.Zona,
    a.flag_consentimiento,
	A.SERVICIO,
	A.AÑO_DURACION_BASE,
	A.MES_DURACION_BASE,
	A.RETIRO,
	cast(a.IMPDHM as numeric(18,2)) as IMPDHM,
    CASE
        WHEN cast(a.IMPDHM as numeric(18,2)) IS NULL THEN '14.SIN DATO'
        WHEN cast(a.IMPDHM as numeric(18,2)) < 10000 THEN '00.[0 - 10,000>'
        WHEN cast(a.IMPDHM as numeric(18,2)) < 20000 THEN '01.[10,000 - 20,000>'
        WHEN cast(a.IMPDHM as numeric(18,2)) < 30000 THEN '02.[20,000 - 30,000>'
        WHEN cast(a.IMPDHM as numeric(18,2)) < 40000 THEN '03.[30,000 - 40,000>'
        WHEN cast(a.IMPDHM as numeric(18,2)) < 50000 THEN '04.[40,000 - 50,000>'
        WHEN cast(a.IMPDHM as numeric(18,2)) < 60000 THEN '05.[50,000 - 60,000>'
        WHEN cast(a.IMPDHM as numeric(18,2)) < 70000 THEN '06.[60,000 - 70,000>'
        WHEN cast(a.IMPDHM as numeric(18,2)) < 80000 THEN '07.[70,000 - 80,000>'
        WHEN cast(a.IMPDHM as numeric(18,2)) < 90000 THEN '08.[80,000 - 90,000>'
        WHEN cast(a.IMPDHM as numeric(18,2)) < 100000 THEN '09.[90,000 - 100,000>'
        WHEN cast(a.IMPDHM as numeric(18,2)) < 150000 THEN '10.[100,000 - 150,000>'
        WHEN cast(a.IMPDHM as numeric(18,2)) < 200000 THEN '11.[150,000 - 200,000>'
        WHEN cast(a.IMPDHM as numeric(18,2)) < 250000 THEN '12.[200,000 - 250,000>'
        WHEN cast(a.IMPDHM as numeric(18,2)) >= 250000 THEN '13.[250,000 A MÁS>'
    END AS RANGO_IMPDHM,
	CASE 
		WHEN a.DEPARTAMENTO IN ('LIMA','CALLAO') THEN 'LIMA' 
		WHEN a.DEPARTAMENTO is null THEN 'SIN DATOS' 
		ELSE 'PROVINCIA' 
	END AS ZONA_PROV,
	CASE	
		WHEN ISNULL(A.[PROVINCIA],'') ='' THEN 'Leads sin región'
		WHEN A.[PROVINCIA] ='LIMA' OR A.[PROVINCIA] ='CALLAO' THEN 'Leads Lima'
		WHEN A.[PROVINCIA] <>'LIMA' THEN 'Leads Provincia' 
	end as REGION,
	CONVERT( VARCHAR(12),A.FECHA_ENVIO,103) AS FECHA_ENVIO,
	CASE WHEN (A.REP1=1 OR A.REP2=1 /*OR A.REP3=1*/) THEN 'Stock' ELSE 'Nuevo' END AS CONJUNTO3MESES,
	case when isnull(B.Estado_,'') ='' then 'NO DISCADO' ELSE B.Estado_ END AS TIPO_GESTION,
	case when isnull(B.Sub_Estado_,'') ='' then 'NO DISCADO' ELSE B.Sub_Estado_ END AS GESTION,
	Case when isnull(B.Descripcion_,'') ='' then 'NO DISCADO' ELSE B.Descripcion_ END AS SUBGESTION,
	B.Dni,
	CASE 
		WHEN ISNULL(FLAT2,0) =0 THEN 'NO ENRIQUECIDO'
		WHEN ISNULL(B.Codigo_Paleta,'')='' THEN  'NO DISCADO' 
		ELSE 'GEST+DISC' END as TIPO,
RECORRIDO=CASE  WHEN B.DNI IS NULL THEN 0 ELSE 1 END,
CASE 
	WHEN ven.ind_venta IS not NULL THEN 1
	WHEN B.Estado_ in ('CONTACTO EFECTIVO CON TITULAR','CONTACTO EFECTIVO') THEN 1 
	ELSE 0
END as CET,	
CASE 
	WHEN ven.ind_venta IS not NULL THEN 1
	WHEN B.Estado_ in ('CONTACTO EFECTIVO CON TITULAR','CONTACTO EFECTIVO') 
		and not Descripcion_ in ('CLIENTE DESEA QUE NO VUELVAN A LLAMAR','NO BRINDA CONSENTIMIENTO','CLIENTE CORTO SIN ESCUCHAR OFERTA')
	THEN 1 ELSE 0 
END as Estado,
b.Hora_Llamada,
CAST(NULL AS VARCHAR(200)) as MARCA2,
cast(null as varchar(200)) AS PERFIL,
b.Mejor_Telefono,
b.PHONE_NUMBER as TELEFONO,
b.Fecha_Llamada AS FECHA_LLAMADA,
substring(convert(varchar,b.Fecha_Llamada),9,2) as DIA,
cast(null as varchar(200)) as Nombre_Campana,
'REGISTROS ENTREGADOS'  AS 'DESCRIPCION',
'REGISTROS CARGADOS'  AS 'DESCRIPCION2',
isnull(isnull(d.tNombreCompleto,b.Ejecutivo),ven.Ejecutivo) as Ejecutivo,
c.NUMXDNI,
ISNULL(d.tSupervisor,'Sin_Asignar') as SUPERVISOR,
FLG_REP_MES_ANT=CASE WHEN ISNULL(REP1,0)=1 THEN 'Stock' Else 'Nuevo' END,
FLG_REP_MESES_ANT=CASE WHEN ISNULL(REP2,0)=1 THEN 'Stock' Else 'Nuevo' END, 
DATENAME(MONTH,GETDATE()) AS tMesGestion,
FLG_SIN_ENR =CASE WHEN ISNULL(FLAT2,0) =0 THEN 'SIN ENRIQUECER' END,
a.PROVINCIA,b.segundos, 
case 
--	WHEN ven.ind_venta IS not NULL THEN 0
	when B.Descripcion_='VOLVER A LLAMAR' then 1 
	else 0 
end as AGENDADOS,
CASE WHEN A.REP1=1 THEN 'Stock' ELSE 'NUEVO' END AS REP1,
CASE WHEN A.REP2=1 THEN 'Stock' ELSE 'NUEVO' END AS REP2,
/*CASE WHEN A.REP3=1 THEN 'Stock' ELSE 'NUEVO' END */
'' AS REP3,
CASE WHEN ven.ind_venta IS NULL THEN 0 ELSE 1 END AS VENTA,
CASE WHEN desem.DNI IS NOT NULL AND desem.Resolucion='TARGET' THEN 1 ELSE 0 END AS VENTA_DESM,
CASE WHEN desem.Resolucion!='TARGET' THEN 1 ELSE 0 END AS VENTA_FUGAS,
desem.MONTO_NETO
FROM [ODIN].[dbo].[Base_Maestra_Efectiva_Negocios_Vigente] a 
full outer join [THOTH].[dbo].[Tmp_LLamadas_Efectiva_Negocios_5] b 
	on a.NUMDOCUMENTO=b.Dni
	and b.Fecha_Llamada>='2026-06-01'
	and b.Fecha_Llamada<'2026-07-01'
full outer join ODIN.dbo.NumxDnixEfectivaN C 
	ON a.NUMDOCUMENTO=c.DNI
full outer join [URANO].[dbo].[tPersonal] d 
	ON b.DNI_EJECUTIVO=d.tDocumentoVici COLLATE Modern_Spanish_CI_AS
full outer join SAMANTHA.dbo.efectiva_negocios_ventas ven 
	ON A.NUMDOCUMENTO=ven.Dni COLLATE Modern_Spanish_CI_AS
	and ven.FECHA>='2026-06-01'
	and ven.FECHA<'2026-07-01'
full outer join SAMANTHA.dbo.efectiva_negocios_ventas_desembolso desem
	ON A.NUMDOCUMENTO=desem.Dni COLLATE Modern_Spanish_CI_AS	
	and desem.FECHA_DESEMBOLSO>='2026-06-01'
	and desem.FECHA_DESEMBOLSO<'2026-07-01'
    """
df_base= pd.read_sql(query, engine_zeus)


In [22]:
dni_duplicados = (
    df_base['NUMERO_DOCUMENTO']
    .value_counts()
    .loc[lambda x: x > 1]
)
dni_duplicados.head()

Series([], Name: count, dtype: int64)

In [21]:
print(
    df_base.loc[
        df_base["NUMERO_DOCUMENTO"].isna(),
        ["NUMERO_DOCUMENTO", "DNI_VEN", "DNI_DES", "DNI_VICI"]
    ].drop_duplicates()
)

       NUMERO_DOCUMENTO DNI_VEN   DNI_DES  DNI_VICI
146870             None    None      None  70666346
146871             None    None      None  47635591
146872             None    None      None  71905919
146873             None    None      None  42287810
146874             None    None      None  10547441
...                 ...     ...       ...       ...
183327             None    None  42538058      None
183328             None    None  09683837      None
183329             None    None  24813547      None
183330             None    None  44787949      None
183331             None    None  41281436      None

[22417 rows x 4 columns]


In [16]:
df_base[['NUMERO_DOCUMENTO','DNI']].head()

KeyError: "['DNI'] not in index"

In [9]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

df_base[df_base['NUMERO_DOCUMENTO'].isna()].head(10)

,NUMERO_DOCUMENTO,Prioridad,TIPOBASEMICROPEQUENA,DEPARTAMENTO,PROVINCIA,DISTRITO,AREAEFECTINEGOCIOS,num_cuotas_pagadas,prioridad,PERFILINICIAL,ULTIMOMONTODESEMBOLSADO,ULTIMOMONTODESEMBOLSADOPLUS20,flag_recurrencia_efectinegocio,RETIRO,DEVUELTO,recompra,Proceso,Zona,flag_consentimiento,SERVICIO,AÑO_DURACION_BASE,MES_DURACION_BASE,RETIRO,IMPDHM,RANGO_IMPDHM,ZONA_PROV,REGION,FECHA_ENVIO,CONJUNTO3MESES,TIPO_GESTION,GESTION,SUBGESTION,Dni,TIPO,RECORRIDO,CET,Estado,Hora_Llamada,MARCA2,PERFIL,Mejor_Telefono,TELEFONO,FECHA_LLAMADA,DIA,Nombre_Campana,DESCRIPCION,DESCRIPCION2,Ejecutivo,NUMXDNI,SUPERVISOR,FLG_REP_MES_ANT,FLG_REP_MESES_ANT,tMesGestion,FLG_SIN_ENR,PROVINCIA,segundos,AGENDADOS,REP1,REP2,REP3,VENTA,VENTA_DESM,VENTA_FUGAS,MONTO_NETO
155586,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,14.SIN DATO,SIN DATOS,Leads sin región,None,Nuevo,NO CONTACTO,SIN CONTACTO,TELEFONO APAGADO,29323519,NO ENRIQUECIDO,1,0,0,12.0,None,None,2.0,958330694,2026-06-05,05,None,REGISTROS ENTREGADOS,REGISTROS CARGADOS,Outbound Auto Dial,NaN,Sin_Asignar,Nuevo,Nuevo,Junio,SIN ENRIQUECER,None,0.0,0,NUEVO,NUEVO,,0,0,0,NaN
155587,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,14.SIN DATO,SIN DATOS,Leads sin región,None,Nuevo,NO CONTACTO,SIN CONTACTO,TELEFONO APAGADO,02701916,NO ENRIQUECIDO,1,0,0,12.0,None,None,1.0,961088910,2026-06-05,05,None,REGISTROS ENTREGADOS,REGISTROS CARGADOS,Outbound Auto Dial,NaN,Sin_Asignar,Nuevo,Nuevo,Junio,SIN ENRIQUECER,None,0.0,0,NUEVO,NUEVO,,0,0,0,NaN
155588,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,14.SIN DATO,SIN DATOS,Leads sin región,None,Nuevo,NO CONTACTO,SIN CONTACTO,TELEFONO FUERA DE SERVICIO/SUSPENDIDO,10072185,NO ENRIQUECIDO,1,0,0,12.0,None,None,1.0,997582108,2026-06-04,04,None,REGISTROS ENTREGADOS,REGISTROS CARGADOS,Outbound Auto Dial,NaN,Sin_Asignar,Nuevo,Nuevo,Junio,SIN ENRIQUECER,None,0.0,0,NUEVO,NUEVO,,0,0,0,NaN
155589,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,14.SIN DATO,SIN DATOS,Leads sin región,None,Nuevo,NO CONTACTO,SIN CONTACTO,TELEFONO FUERA DE SERVICIO/SUSPENDIDO,41704283,NO ENRIQUECIDO,1,0,0,12.0,None,None,1.0,997897845,2026-06-04,04,None,REGISTROS ENTREGADOS,REGISTROS CARGADOS,Outbound Auto Dial,NaN,Sin_Asignar,Nuevo,Nuevo,Junio,SIN ENRIQUECER,None,0.0,0,NUEVO,NUEVO,,0,0,0,NaN
155590,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,14.SIN DATO,SIN DATOS,Leads sin región,None,Nuevo,NO CONTACTO,SIN CONTACTO,TELEFONO APAGADO,40146520,NO ENRIQUECIDO,1,0,0,11.0,None,None,1.0,950346075,2026-06-05,05,None,REGISTROS ENTREGADOS,REGISTROS CARGADOS,Outbound Auto Dial,NaN,Sin_Asignar,Nuevo,Nuevo,Junio,SIN ENRIQUECER,None,0.0,0,NUEVO,NUEVO,,0,0,0,NaN
155591,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,14.SIN DATO,SIN DATOS,Leads sin región,None,Nuevo,NO CONTACTO,SIN CONTACTO,TONO OCUPADO,10708172,NO ENRIQUECIDO,1,0,0,10.0,None,None,1.0,989866672,2026-06-05,05,None,REGISTROS ENTREGADOS,REGISTROS CARGADOS,Outbound Auto Dial,NaN,Sin_Asignar,Nuevo,Nuevo,Junio,SIN ENRIQUECER,None,0.0,0,NUEVO,NUEVO,,0,0,0,NaN
155661,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,14.SIN DATO,SIN DATOS,Leads sin región,None,Nuevo,NO CONTACTO,SIN CONTACTO,TELEFONO APAGADO,41897609,NO ENRIQUECIDO,1,0,0,10.0,None,None,1.0,995715184,2026-06-05,05,None,REGISTROS ENTREGADOS,REGISTROS CARGADOS,Outbound Auto Dial,NaN,Sin_Asignar,Nuevo,Nuevo,Junio,SIN ENRIQUECER,None,0.0,0,NUEVO,NUEVO,,0,0,0,NaN
155662,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,14.SIN DATO,SIN DATOS,Lead

In [ ]:
SELECT 
1 AS NRG,
A.NUMDOCUMENTO as NUMERO_DOCUMENTO,
a.SEGMICROPEQUENACOMERCIAL as Prioridad,
a.IMPDHM,
    CASE
        WHEN a.IMPDHM IS NULL THEN 'SIN DATO'
        WHEN a.IMPDHM < 10000 THEN '00.<0 - 10,000]'
        WHEN a.IMPDHM < 20000 THEN '01.<10,000 - 20,000]'
        WHEN a.IMPDHM < 30000 THEN '02.<20,000 - 30,000]'
        WHEN a.IMPDHM < 40000 THEN '03.<30,000 - 40,000]'
        WHEN a.IMPDHM < 50000 THEN '04.<40,000 - 50,000]'
        WHEN a.IMPDHM < 60000 THEN '05.<50,000 - 60,000]'
        WHEN a.IMPDHM < 70000 THEN '06.<60,000 - 70,000]'
        WHEN a.IMPDHM < 80000 THEN '07.<70,000 - 80,000]'
        WHEN a.IMPDHM < 90000 THEN '08.<80,000 - 90,000]'
        WHEN a.IMPDHM < 100000 THEN '09.<90,000 - 100,000]'
        WHEN a.IMPDHM < 150000 THEN '10.<100,000 - 150,000]'
        WHEN a.IMPDHM < 200000 THEN '11.<150,000 - 200,000]'
        WHEN a.IMPDHM < 250000 THEN '12.<200,000 - 250,000]'
        WHEN a.IMPDHM > 250000 THEN '13.<250,000 A MÁS]'
    END AS RANGO_IMPDHM,
	a.TIPOBASEMICROPEQUENA,
    a.DEPARTAMENTO,
	CASE 
		WHEN a.DEPARTAMENTO IN ('LIMA','CALLAO') THEN 'LIMA' 
		WHEN a.DEPARTAMENTO is null THEN 'SIN DATOS' 
		ELSE 'PROVINCIA' 
	END AS ZONA_PROV,
	CASE	
		WHEN ISNULL(A.[PROVINCIA],'') ='' THEN 'Leads sin región'
		WHEN A.[PROVINCIA] ='LIMA' OR A.[PROVINCIA] ='CALLAO' THEN 'Leads Lima'
		WHEN A.[PROVINCIA] <>'LIMA' THEN 'Leads Provincia' 
	end as REGION,
    a.PROVINCIA,
    a.DISTRITO,
	CONVERT( VARCHAR(12),A.FECHA_ENVIO,103) AS FECHA_ENVIO,
    a.AREAEFECTINEGOCIOS,
    a.num_cuotas_pagadas,
    a.num_cuotas_impagas,
    a.prioridad,
    a.PERFILINICIAL,
    a.ULTIMOMONTODESEMBOLSADO,
    a.ULTIMOMONTODESEMBOLSADOPLUS20,
    a.flag_recurrencia_efectinegocio,
    a.RETIRO,
    a.DEVUELTO,
    a.recompra,
    a.Proceso, --MARCA
    a.Zona,
    a.flag_consentimiento,
	A.SERVICIO,
	A.AÑO_DURACION_BASE,
	A.MES_DURACION_BASE,
	A.RETIRO,
CASE WHEN (A.REP1=1 OR A.REP2=1 /*OR A.REP3=1*/) THEN 'Stock' ELSE 'Nuevo' END AS CONJUNTO3MESES,
case when isnull(B.Estado_,'') ='' then 'NO DISCADO' ELSE B.Estado_ END AS TIPO_GESTION,
case when isnull(B.Sub_Estado_,'') ='' then 'NO DISCADO' ELSE B.Sub_Estado_ END AS GESTION,
Case when isnull(B.Descripcion_,'') ='' then 'NO DISCADO' ELSE B.Descripcion_ END AS SUBGESTION,
B.Dni,
'TIPO'=CASE 
		WHEN ISNULL(FLAT2,0) =0 THEN 'NO ENRIQUECIDO'
		WHEN ISNULL(B.Codigo_Paleta,'')='' THEN  'NO DISCADO' 
		ELSE 'GEST+DISC' END,
RECORRIDO=CASE  WHEN B.DNI IS NULL THEN 0 ELSE 1 END,
CASE WHEN B.Estado_ in ('CONTACTO EFECTIVO CON TITULAR','CONTACTO EFECTIVO') THEN 1 ELSE 0 END as CET,	
CASE 
	WHEN B.Estado_ in ('CONTACTO EFECTIVO CON TITULAR','CONTACTO EFECTIVO') 
				and not Descripcion_ in ('CLIENTE DESEA QUE NO VUELVAN A LLAMAR','NO BRINDA CONSENTIMIENTO','CLIENTE CORTO SIN ESCUCHAR OFERTA')
	THEN 1 ELSE 0 
END as CONT_GEN,
b.Hora_Llamada,
CAST(NULL AS VARCHAR(200)) as MARCA2,
cast(null as varchar(200)) AS PERFIL,
b.Mejor_Telefono,
b.PHONE_NUMBER as TELEFONO,
b.Fecha_Llamada AS FECHA_LLAMADA,
substring(convert(varchar,b.Fecha_Llamada),9,2) as DIA,
cast(null as varchar(200)) as Nombre_Campana,
'REGISTROS ENTREGADOS'  AS 'DESCRIPCION',
'REGISTROS CARGADOS'  AS 'DESCRIPCION2',
isnull(d.tNombreCompleto,b.Ejecutivo) as Ejecutivo,
c.NUMXDNI,
ISNULL(d.tSupervisor,'Sin_Asignar') as SUPERVISOR,
FLG_REP_MES_ANT=CASE WHEN ISNULL(REP1,0)=1 THEN 'Stock' Else 'Nuevo' END,
FLG_REP_MESES_ANT=CASE WHEN ISNULL(REP2,0)=1 THEN 'Stock' Else 'Nuevo' END, 
DATENAME(MONTH,GETDATE()) AS tMesGestion,
FLG_SIN_ENR =CASE WHEN ISNULL(FLAT2,0) =0 THEN 'SIN ENRIQUECER' END,
a.PROVINCIA,b.segundos, 
case 
	when 
		B.Estado_ in ('CONTACTO EFECTIVO CON TITULAR','CONTACTO EFECTIVO') 
		and not Descripcion_ in ('CLIENTE DESEA QUE NO VUELVAN A LLAMAR','NO BRINDA CONSENTIMIENTO','CLIENTE CORTO SIN ESCUCHAR OFERTA') 
	then 1 
	else 0 
end AS VAL,
case when B.Descripcion_='VOLVER A LLAMAR' then 1 else 0 end as AGENDADOS,
cast(null as varchar(100)) Estado,
cast(null as NUMERIC) CntEstado,
cast(null as NUMERIC) MntOferta,
cast(null as int) CNTVTAS,
cast(null as int)[NUM_DIA_HABIL],
cast(null as varchar(100)) [Semana_Mes],
cast(null as float) tMontoDesem,
CASE WHEN A.REP1=1 THEN 'Stock' ELSE 'NUEVO' END AS REP1,
CASE WHEN A.REP2=1 THEN 'Stock' ELSE 'NUEVO' END AS REP2,
/*CASE WHEN A.REP3=1 THEN 'Stock' ELSE 'NUEVO' END */
'' AS REP3,
CASE WHEN ven.ind_venta IS NULL THEN 0 ELSE 1 END AS VENTA,
CASE WHEN desem.DNI IS NOT NULL AND desem.Resolucion='TARGET' THEN 1 ELSE 0 END AS VENTA_DESM,
CASE WHEN desem.Resolucion!='TARGET' THEN 1 ELSE 0 END AS VENTA_FUGAS,
desem.MONTO_NETO
FROM [ODIN].[dbo].[Base_Maestra_Efectiva_Negocios_Vigente] a 
FULL OUTER JOIN [THOTH].[dbo].[Tmp_LLamadas_Efectiva_Negocios_5] b 
on a.NUMDOCUMENTO=b.Dni
FULL OUTER JOIN ODIN.dbo.NumxDnixEfectivaN C 
ON a.NUMDOCUMENTO=c.DNI
FULL OUTER JOIN [URANO].[dbo].[tPersonal] d 
ON b.DNI_EJECUTIVO=d.tDocumentoVici COLLATE Modern_Spanish_CI_AS
FULL OUTER JOIN SAMANTHA.dbo.efectiva_negocios_ventas ven 
ON A.NUMDOCUMENTO=B.Dni
FULL OUTER JOIN SAMANTHA.dbo.efectiva_negocios_ventas_desembolso desem
ON A.NUMDOCUMENTO=B.Dni



/*
SELECT *
FROM sys.synonyms
WHERE name = 'synNumeroEfectivaNegocios';

*/



In [ ]:
# def since_base_maestra_efe_negocio(spark):
query = """
    Select  
    ROW_NUMBER() OVER (ORDER BY (SELECT NULL)) AS indice,
    ISNULL([SEGMICROPEQUENACOMERCIAL],'IMPULSA') as first_name,  
    isnull([DEPARTAMENTO],'Sin_Departamento') as last_name,  
    CASE 
        WHEN Materno IS NULL THEN Nombres +' '+Paterno 
        ELSE Nombres + ' ' + Paterno +' '+ Materno 
    END AS address1,  
    [DIRECCION] as address2,  
    'EMP1='+[empresa1]+'/ EMP2='+[empresa2]+'/ EMP3='+ [empresa3] as address3,  
    Proceso as title,  
    Montos_Referenciales as comments, 
    'IMPORTE DEUDA'+' '+CONVERT(VARCHAR,IMPDHM) as security_phrase,  
    NUMDOCUMENTO as vendor_lead_code,  
    PROVINCIA as province,  
    DISTRITO as city,  
    AREAEFECTINEGOCIOS AS email,  
    IMPDHM AS deuda,  
    SEGMICROPEQUENACOMERCIAL as tip_prioridad,  
    Zona,  
    perfil_ic,  
    canal_asignado,
    CASE 
        WHEN REP1=1 then 'STOCK' 
        ELSE 'NUEVO'
    END as marca,  
    fecha_envio,      
    TIPOBASEMICROPEQUENA,     
    case
        when len(replace(retiro,' ',''))>3  then lower(replace(retiro,' ','_'))
        else 'no_aplica'
    end as retiro 
    from dantalion.dbo.Base_Maestra_Efectiva_Negocios_Vigente

select tDocumento,tNombreCompleto,tSexo,tGrupo,tCargo,tCampana,
tSupervisor,tEmpresa,tContrato,tModalidad,tCondicion,tHorario,
tCese,tEstado,tNivel1
    from [URANO].[dbo].[tPersonal]

    """
        
df_base= obtener_tabla_sql(spark,query,server_kishin,user_kishin,pwd_kishin,db_kishin)

df_base = df_base.withColumn(
    "region",
    F.when(F.coalesce(F.col("province"), F.lit("")) == "", "Leads sin región")
    .when(F.col("province").isin("LIMA", "CALLAO"), "Leads Lima")
    .otherwise("Leads Provincia")
    )

    df_base = df_base.withColumn(
        "prioridada",
        F.when(
            (F.col("province").isin("LIMA", "CALLAO")) &
            (F.col("title") == "humano seguro") &
            (F.col("tip_prioridad") == "prf"),
            "prioridad 2"
        )
        .when(
            (F.col("province") != "LIMA") &
            (F.col("title") == "fasttrack") &
            (F.col("tip_prioridad") == "elt"),
            "prioridad 1"
        )
        .when(
            (F.col("province").isin("LIMA", "CALLAO")) &
            (F.col("title") == "FULL") &
            (F.col("tip_prioridad") == "prf"),
            "prioridad 3"
        )
        .otherwise("otra prioridad")
    )

    return df_base.withColumn(
        "vendor_lead_code",
        F.right(
            F.concat(F.lit("00000000"), F.col("vendor_lead_code")),
            F.lit(8)
        )
    )
    # print(df_base.columns)


In [ ]:

    return pd.read_sql(query, engine_sa)
    return pd.read_sql(query, engine_sa)
    return pd.read_sql(query, engine_sa)

In [ ]:

def since_target_sales(engine_zeus,fecha_mes_base,campana):
    query = f"""
        select FECHA as fecha_gestion,DNI as vendor_lead_code,
        PROMOTOR as promotor,
        ESTADO as estado_venta,
        TRAMA_HORA as tramo_venta,
        MONTO as monto_venta,
        DNIEjecutivo as dni_ejecutivo_venta,
        Producto as producto_venta,
        Producto as title,
        Celular as cel_venta,1 as venta,
        subcampana
        from SAMANTHA.dbo.Ventas_Target
        where campana='{campana}'
        and fecha >= DATEADD(MONTH, -4, DATEADD(DAY, 1, EOMONTH('{fecha_mes_base}'))) 
        and fecha < DATEADD(MONTH, 0, DATEADD(DAY, 1, EOMONTH('{fecha_mes_base}')))
        """
    return pd.read_sql(query, engine_zeus)

def since_target_disbursements(engine_sa,fecha_mes_base,campana):
    query = f"""
        select cast(FECHA_HORA_DESEMBOLSO as date) as fecha_desembolso,
        CAST(FECHA_HORA_DESEMBOLSO AS TIME) AS hora_desembolso,
        cast(TARGET_23_FG as date) as target_23_fg,
        CAST(TARGET_23_FG AS TIME) AS target_23_hg,
        DNI as vendor_lead_code,
        target_23_cg,
        target_23_cg,
        TARGET_23_OBS,
        tipo,
        desbase,
        monto_neto,
        monto_bruto,
        prestamo,
        canal_confirmado,
        perfil
        from VALENTINA.dbo.efectiva_ventas_desembolso
        where campana='{campana}'
        and FECHA_HORA_DESEMBOLSO >= DATEADD(MONTH, -4, DATEADD(DAY, 1, EOMONTH('{fecha_mes_base}'))) 
        and FECHA_HORA_DESEMBOLSO < DATEADD(MONTH, 0, DATEADD(DAY, 1, EOMONTH('{fecha_mes_base}')))
        """
    return pd.read_sql(query, engine_sa)





In [5]:
df_venta=since_sales(engine_zeus,fecha_mes_base,campana)

In [ ]:
df_venta=since_sales(engine_zeus,fecha_mes_base,campana)
